# 3D Medical CLIP — CT-RATE Retrieval Training (P12, Colab)

Cloud training for the PointNet++ (CT) + Bio_ClinicalBERT (report) retrieval
model. Reuses the **exact validated training code** from the repo
(`ml/models/train.py`, `data.py`, `ml/preprocessing/`) — this notebook only
orchestrates data acquisition, preprocessing, and the training run.

**Why cloud:** the from-scratch PointNet++ CT encoder needs far more data than a
local subset provides. Local runs (≤556 volumes) memorize the training set but
generalize at ~random on held-out data. This notebook trains on a larger CT-RATE
subset (configurable `TARGET_VOLUMES`).

### Prerequisites
1. **Runtime → Change runtime type → GPU** (T4 is fine; A100/L4 faster).
2. A free **Hugging Face account** with a **read token** — https://huggingface.co/settings/tokens
3. **Accept the CT-RATE dataset terms** (one click) — https://huggingface.co/datasets/ibrahimhamamci/CT-RATE
4. A **Google Drive** with a few GB free (persistent point-cloud cache + checkpoints;
   lets training resume across Colab disconnects).


## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime > Change runtime type > GPU'

## 2. Install dependencies
Colab ships CUDA torch; we add the ML/data deps at repo-pinned versions.

In [ ]:
!pip -q install nibabel==5.4.2 scipy transformers==5.13.1 'huggingface_hub>=1.20' hf_transfer
print('deps installed')

## 3. Clone the repo
Public repo; contains the validated training pipeline.

In [ ]:
import os, sys
REPO_URL = 'https://github.com/Sir7s/MedicalCLAP.git'  # <-- your repo
BRANCH   = 'phase/P12-training'  # P12 code; change to 'main' once P12 is merged
if not os.path.isdir('/content/MedicalCLAP'):
    !git clone --depth 1 -b $BRANCH $REPO_URL /content/MedicalCLAP
os.chdir('/content/MedicalCLAP')
sys.path.insert(0, '/content/MedicalCLAP')
print('cwd', os.getcwd(), '| branch', BRANCH)

## 4. Mount Google Drive (persistent cache + checkpoints)
Point clouds and checkpoints live on Drive so a disconnect doesn't lose work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/medclip_p12'
os.makedirs(DRIVE + '/cache', exist_ok=True)
os.makedirs(DRIVE + '/runs', exist_ok=True)
os.environ['MEDCLIP_CACHE_DIR'] = DRIVE + '/cache'     # point-cloud cache on Drive
os.environ['MEDCLIP_DATA_ROOT'] = 'data/ct_rate'       # raw volumes on Colab disk
os.environ['HF_HUB_DISABLE_XET'] = '1'
RUN_DIR = DRIVE + '/runs/p12_cloud'
print('cache ->', os.environ['MEDCLIP_CACHE_DIR'])
print('runs  ->', RUN_DIR)

## 5. Hugging Face token
Pasted into the git-ignored `infra/.env` that the P7 downloader reads. Not logged.

In [ ]:
import getpass, pathlib
tok = getpass.getpass('HF read token (hf_...): ').strip()
pathlib.Path('infra').mkdir(exist_ok=True)
pathlib.Path('infra/.env').write_text('HF_TOKEN=' + tok + '\n', encoding='utf-8')
os.environ['HF_TOKEN'] = tok
print('wrote infra/.env')

## 6. Training configuration
`TARGET_VOLUMES` is the total CT-RATE subset size (patient-level split → ~70% train).
Bigger = better generalization but longer download/preprocess. Start with 3000
(~2000 train); raise toward the full ~20k train set with more compute/time.

| GPU | suggested batch | points |
|-----|-----------------|--------|
| T4 (16 GB)  | 8  | 32768 |
| A100 (40 GB)| 24 | 32768 |


In [ ]:
TARGET_VOLUMES = 3000     # total volumes across train/val/test
POINTS         = 32768    # spec full-scale target
BATCH          = 8
EPOCHS         = 40
FREEZE_TEXT    = True     # set False (full fine-tune BERT) only for large TARGET_VOLUMES
print(dict(TARGET_VOLUMES=TARGET_VOLUMES, POINTS=POINTS, BATCH=BATCH, EPOCHS=EPOCHS, FREEZE_TEXT=FREEZE_TEXT))

## 7. Download metadata + build a patient-level split
Small CSVs (reports + labels) drive the split and supervision. Zero patient leakage.

In [ ]:
from huggingface_hub import hf_hub_download
REPO = 'ibrahimhamamci/CT-RATE'
meta = {
  'dataset/radiology_text_reports/train_reports.csv': None,
  'dataset/multi_abnormality_labels/train_predicted_labels.csv': None,
  'dataset/metadata/no_chest_train.txt': None,
}
for rp in list(meta):
    try:
        hf_hub_download(REPO, rp, repo_type='dataset', local_dir='data/ct_rate',
                        token=os.environ['HF_TOKEN'])
        print('ok', rp)
    except Exception as e:
        print('skip', rp, '—', e)

In [ ]:
from ml.datasets.ct_rate.select import build_split, write_split
rev = build_split(target_volumes=TARGET_VOLUMES, seed=42)
write_split(rev)
print('split counts:', rev.counts, '| patients:', rev.patient_counts)

## 8. Download the volumes
Resumable (skips already-present files). This is the slowest step; for large
`TARGET_VOLUMES` it may span multiple Colab sessions — just re-run; it resumes.

In [ ]:
from ml.datasets.ct_rate.acquire import download_subset
summary = download_subset()   # downloads every volume named in the split
print(summary)

## 9. Preprocess → point-cloud cache on Drive
Deterministic P9 pipeline. Idempotent: cached `.npz` on Drive are skipped, so this
resumes across sessions and (once done) training no longer needs the raw volumes.

In [ ]:
from ml.models.data import select_subset, prepare_cache
from ml.models.train_config import TrainConfig, DEFAULT
cfg = TrainConfig(**{**DEFAULT.to_dict(), 'n_train': rev.counts['train'],
                     'n_val': rev.counts['val'], 'n_test': rev.counts['test'],
                     'n_points': POINTS})
subset = select_subset(cfg)
prepare_cache(cfg, subset)   # writes .npz to MEDCLIP_CACHE_DIR (Drive)
print('cache ready for', {k: len(v) for k, v in subset.items()})

## 10. Train (resumable)
Reuses `ml/models/train.py`. Saves `best.pt` (val-selected) + `last.pt` (resume) +
`metrics.json` + `model_card.md` to the Drive run dir. Re-run this cell after any
disconnect — `--resume` continues from `last.pt`.

In [ ]:
freeze_flag = '' if FREEZE_TEXT else '--no-freeze'
!python -m ml.models.train \
    --out "$RUN_DIR" --resume \
    --n-train {rev.counts['train']} --n-val {rev.counts['val']} --n-test {rev.counts['test']} \
    --points {POINTS} --batch {BATCH} --epochs {EPOCHS} {freeze_flag}

## 11. Results

In [ ]:
import json
m = json.load(open(RUN_DIR + '/metrics.json'))
print('scope:', m['scope'])
print('best epoch:', m['best_epoch'])
print('TEST:', {k: round(v, 3) for k, v in m['test'].items()})
print('random R@1 =', round(1/m['counts']['test'], 4))
print(open(RUN_DIR + '/model_card.md').read())

## 12. Download the checkpoint
Hand `best.pt` + `metrics.json` + `model_card.md` back to the project to finalize
P12. (Weights are **not** committed to git — they load into the running app.)

In [ ]:
import shutil
shutil.make_archive('/content/p12_cloud_artifacts', 'zip', RUN_DIR)
from google.colab import files
files.download('/content/p12_cloud_artifacts.zip')
print('also saved on Drive at', RUN_DIR)